Import tensorflow and make sure that it works in your environment.
The dependencies include:
    - tensorflow
    - torch
    - GitPython
    - transformers
    - datasets
    - evaluate
    - lime
    - accelerate

In [41]:
%pip install tensorflow torch GitPython transformers datasets evaluate lime accelerate

In [2]:
import tensorflow as tf
print(tf.__version__)

2.21.0


In [57]:
import torch
import git
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding
import datasets
import evaluate
from lime.lime_text import LimeTextExplainer
from pathlib import Path
import accelerate


Get data from kaggle.

In [58]:
data_dir = Path("medical_nlp")
if not data_dir.is_dir():
    url = "https://github.com/salgadev/medical-nlp.git"
    local_repo = "./medical_nlp"
    git.Repo.clone_from(url,local_repo)

Load data into dataset.

In [61]:
training_dataset = datasets.load_dataset("./medical_nlp/data")

Or you could just run the following to get pre-defined data.

In [62]:
#uncomment to load directly
#training_dataset = datasets.load_dataset("medical_nlp")

Get models from huggingface

In [67]:
id2label = {1: "Surgery", 2: "Medical Records",3:"Internal Medicine",4:"Other"}
label2id = {"Surgery": 1, "Medical Records": 2,"Internal Medicine":3,"Other":4}

model_file = Path("./model.safetensors")
tokenizer = None
model = None
if not model_file.is_file:
     tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-cased")
     tokenizer.save_pretrained()
     model = AutoModelForSequenceClassification.from_pretrained(
         "google-bert/bert-base-cased",
         num_labels=4,
         id2label=id2label,
         label2id=label2id
     )   
         
     model.save_pretrained(".")
else:
     tokenizer = AutoTokenizer.from_pretrained(".")
     model = AutoModelForSequenceClassification.from_pretrained(
         ".",
         num_labels=4,
         id2label=id2label,
         label2id=label2id
     )

device = torch.device("cpu")
model.to(device)
model.eval()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: .
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,),

Tokenize training data

In [68]:
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding=False)
tokenized_datasets = training_dataset.map(tokenize_function, batched=True)


Setup Compute Metrics for Training

In [69]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)



Train categorization model

In [70]:
model_dir = Path("medical_nlp_text_classifier")

#don't retrain if model is already there.
if not model_dir.is_dir():
    training_args = TrainingArguments(
        output_dir="my_text_classifier",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=2,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["test"],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
 
    trainer.train()
    trainer.save_model("./medical_nlp_text_classifier")

Test case

In [71]:
TEST = "PREOPERATIVE DIAGNOSIS: , Lumbar stenosis.,POSTOPERATIVE DIAGNOSES:,  Lumbar stenosis and cerebrospinal fluid fistula.,TITL    E OF THE OPERATION,1.  Lumbar laminectomy for decompression with foraminotomies L3-L4, L4-L5, L5-S1 microtechniques.,2.  Repair of     CSF fistula, microtechniques L5-S1, application of DuraSeal.,INDICATIONS:,  The patient is an 82-year-old woman who has about a fou    r-month history now of urinary incontinence and numbness in her legs and hands, and difficulty ambulating.  She was evaluated with     an MRI scan, which showed a very high-grade stenosis in her lumbar spine, and subsequent evaluation included a myelogram, which dem    onstrated cervical stenosis at C4-C5, C5-C6, and C6-C7 as well as a complete block of the contrast at L4-L5 and no contrast at L5-S    1 either and stenosis at L3-L4 and all the way up, but worse at L3-L4, L4-L5, and L5-S1.  Yesterday, she underwent an anterior cerv    ical discectomy and fusions C4-C5, C5-C6, C6-C7 and had some improvement of her symptoms and increased strength, even in the recove    ry room.  She was kept in the ICU because of her age and the need to bring her back to the operating room today for decompressive l    umbar laminectomy."

Load and run classifier model

In [72]:
classifier = pipeline("text-classification", model="./medical_nlp_text_classifier")
result = classifier(TEST)
print(result)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[{'label': 'Surgery', 'score': 0.8707497715950012}]


Setup Lime

In [73]:
def predict_probability(texts):
    inputs = tokenizer(
        texts, 
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=1)

    return probs.cpu().numpy()

class_names = ["Surgery",
            "Medical Records",
            "Internal Medicine",
            "Other"]

Run Lime

In [ ]:
explainer = LimeTextExplainer(class_names=class_names)
exp = explainer.explain_instance(TEST,predict_probability)

print(exp.show_in_notebook())